In [1]:
from Ingestor import Ingestor
from embedder import Embedder
from retriever import Retriever
from vector_db import VectorStore

c:\Users\Lenovo\Desktop\Rag_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def evidence_for(answers):
    """Text evidence paragraphs for one question (tables and figures excluded)."""
    evidence = set()
    for answer in answers["answer"]:
        for e in answer["evidence"]:
            if not e.startswith("FLOAT SELECTED"):
                evidence.add(e)
    return evidence


def run_retrieval(dataset, num_papers, ret, reranker=None,
                  methods=("dense", "keyword", "hybrid", "hybrid+rerank"), k=20):
    records = {method: [] for method in methods}

    for paper in dataset.select(range(num_papers)):
        qas = paper["qas"]
        for question, answers in zip(qas["question"], qas["answers"]):
            evidence = evidence_for(answers)
            if not evidence:
                continue

            cache = {}  # reuse the hybrid results for "hybrid" and "hybrid+rerank"

            for method in methods:
                if method == "hybrid+rerank":
                    if "hybrid" not in cache:
                        cache["hybrid"] = ret.search(question, k=k, paper_id=paper["id"], method="hybrid")
                    chunks = reranker.rerank(question, cache["hybrid"], top_k=k)  # reorder all k, don't drop any
                else:
                    chunks = ret.search(question, k=k, paper_id=paper["id"], method=method)
                    if method == "hybrid":
                        cache["hybrid"] = chunks

                correct_ranks = [
                    rank for rank, chunk in enumerate(chunks, start=1)
                    if chunk["text"] in evidence
                ]
                records[method].append({
                    "correct_ranks": correct_ranks,
                    "n_evidence": len(evidence),
                })

    return records


In [3]:
def compute_metrics(question_records, ks=(1, 5, 10, 20)):
    """Metrics at several cutoffs, from records retrieved with k = max(ks)."""
    n = len(question_records)
    results = {"questions": n}

    for k in ks:
        hit = mrr = recall = precision = 0.0
        for r in question_records:
            ranks = [rank for rank in r["correct_ranks"] if rank <= k]  # only what's in the top k
            if ranks:
                hit += 1
                mrr += 1 / ranks[0]
            recall += len(ranks) / r["n_evidence"]
            precision += len(ranks) / k

        results[f"hit@{k}"] = hit / n
        results[f"MRR@{k}"] = mrr / n
        results[f"recall@{k}"] = recall / n
        results[f"precision@{k}"] = precision / n

    return results

In [4]:
ingestor = Ingestor()
embedder = Embedder()
vectorstore = VectorStore(dimension=embedder.dimension)
ret = Retriever(embedder, vectorstore)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 273.28it/s]
c:\Users\Lenovo\Desktop\Rag_Project\embedder.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


In [5]:
# ingestor.ingest(embedder, vectorstore) 
print(vectorstore.count())

{'chunks': 13266, 'vec_chunks': 13266, 'fts_chunks': 13266}


In [6]:
ks = (1,3, 5, 10, 20)

dataset = ingestor.load_qasper_validation()

from reranker import Reranker  # or wherever you define it

reranker = Reranker()  # create once

# Start small to measure the time: reranking is slow on CPU
records = run_retrieval(dataset, 20, ret, reranker=reranker, k=max(ks))

for method, question_records in records.items():
    m = compute_metrics(question_records, ks)
    print(f"\n{method}  ({m['questions']} questions)")
    for k in ks:
        print(f"  @{k:<3} hit={m[f'hit@{k}']:.3f}  MRR={m[f'MRR@{k}']:.3f}  "
              f"recall={m[f'recall@{k}']:.3f}  precision={m[f'precision@{k}']:.3f}")

Loading weights: 100%|██████████| 393/393 [00:04<00:00, 87.90it/s] 



dense  (52 questions)
  @1   hit=0.308  MRR=0.308  recall=0.179  precision=0.308
  @3   hit=0.558  MRR=0.413  recall=0.318  precision=0.212
  @5   hit=0.712  MRR=0.446  recall=0.448  precision=0.188
  @10  hit=0.885  MRR=0.470  recall=0.604  precision=0.129
  @20  hit=0.962  MRR=0.476  recall=0.831  precision=0.097

keyword  (52 questions)
  @1   hit=0.192  MRR=0.192  recall=0.138  precision=0.192
  @3   hit=0.442  MRR=0.301  recall=0.239  precision=0.154
  @5   hit=0.654  MRR=0.348  recall=0.363  precision=0.158
  @10  hit=0.846  MRR=0.375  recall=0.586  precision=0.129
  @20  hit=0.923  MRR=0.380  recall=0.790  precision=0.091

hybrid  (52 questions)
  @1   hit=0.288  MRR=0.288  recall=0.139  precision=0.288
  @3   hit=0.615  MRR=0.423  recall=0.310  precision=0.218
  @5   hit=0.731  MRR=0.448  recall=0.447  precision=0.185
  @10  hit=0.865  MRR=0.465  recall=0.578  precision=0.123
  @20  hit=1.000  MRR=0.475  recall=0.870  precision=0.099

hybrid+rerank  (52 questions)
  @1   hit=0

In [7]:
from reranker import Reranker

In [8]:
# testing the generator on 1 question

from generator import Generator

generator = Generator()


In [9]:

question = "What language pairs are explored?"
paper_id = "1912.01214"

chunks = ret.search(question, k=5, paper_id=paper_id)

answer, stats = generator.generate(question, chunks)

print("Answer:", answer)
print(stats)

# print("Sections used:", [c["section"] for c in chunks])

Answer: The following language pairs are explored:  
- French-English-Spanish (Fr-En-Es)  
- German-English-French (De-En-Fr)  
- Romanian-English-German (Ro-En-De)  
- Arabic-English-Spanish (Ar-En-Es)  
- Arabic-English-Russian (Ar-En-Ru)  
- Spanish-English-Russian (Es-En-Ru)  

These pairs are evaluated using English as the pivot language, with zero-shot translation settings [Section: Experiments ::: Setup ::: Datasets].
{'load_s': 34.3430189, 'prompt_tokens': 954, 'prefill_s': 76.876533, 'output_tokens': 120, 'decode_s': 35.560057, 'tok_per_s': 3.3745727685419626}


In [10]:
print(generator.model)


qwen3:4b-instruct-2507-q4_K_M


In [11]:
import time, httpx

start = time.perf_counter()
r = httpx.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b-instruct-2507-q4_K_M",
        "messages": [{"role": "user", "content": "Say hello in one sentence."}],
        "stream": False,
        "keep_alive": "30m",
        "options": {"num_ctx": 4096, "num_predict": 50},
    },
    timeout=None,
)
data = r.json()
print(f"wall time: {time.perf_counter() - start:.1f}s", data["message"]["content"])
print("load     :", data["load_duration"] / 1e9, "s")
print("prefill  :", data["prompt_eval_count"], "tokens in", data["prompt_eval_duration"] / 1e9, "s")
print("decode   :", data["eval_count"], "tokens in", data["eval_duration"] / 1e9, "s")
print("decode speed:", data["eval_count"] / (data["eval_duration"] / 1e9), "tok/s")

wall time: 5.9s Hello! 😊
load     : 0.0713413 s
prefill  : 14 tokens in 1.308828 s
decode   : 5 tokens in 0.954809 s
decode speed: 5.2366494241256625 tok/s
